# FT5009 Individual Assignment 1
## Reading the Market: Data Forensics & Quality

Student Name: Shu Zixuan
Student ID: [redacted]

# 1. Setup and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

from pathlib import Path
from scipy.stats import chisquare

In [ ]:
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("yfinance:", yf.__version__)

# 2. Data Collection and Raw Data Cache

Three liquid U.S. stocks are selected for the main analysis:

- AAPL (Apple)
- JPM (JPMorgan Chase)
- XOM (Exxon Mobil)

The three stocks provide exposure to technology, financials, and energy while keeping the trading calendar and reporting currency consistent across the sample.

AMZN serves as the corporate-action case because its 20-for-1 stock split falls within the sample window and provides a clear setting for comparing raw and adjusted prices.

The sample covers daily market data from 1 September 2016 to 1 September 2026. Data are retrieved through yfinance with `auto_adjust=False`, preserving both raw Close and Adjusted Close. Each original download is cached as a parquet file and remains unchanged throughout the analysis; the submitted grading path reads these caches without network access, and all subsequent processing operates on separate working copies.

**Data source and retrieval record**

- Provider: [Yahoo Finance](https://finance.yahoo.com/) via the [yfinance Python package](https://github.com/ranaroussi/yfinance).
- Historical-data pages: [AAPL](https://finance.yahoo.com/quote/AAPL/history/), [JPM](https://finance.yahoo.com/quote/JPM/history/), [XOM](https://finance.yahoo.com/quote/XOM/history/), and [AMZN](https://finance.yahoo.com/quote/AMZN/history/).
- The parquet cache was inspected on 5 September 2026; the submitted grading path reads these files without making network requests.


In [ ]:
MAIN_TICKERS = ["AAPL", "JPM", "XOM"]
SPLIT_TICKER = "AMZN"

ALL_TICKERS = MAIN_TICKERS + [SPLIT_TICKER]

START_DATE = "2016-09-01"
END_DATE = "2026-09-02"

print("Main stocks:", MAIN_TICKERS)
print("Split stock:", SPLIT_TICKER)
print("Download window:", START_DATE, "to", END_DATE)

In [ ]:
cwd = Path.cwd()

if cwd.name == "code":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

print("Current working directory:", cwd)
print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

In [ ]:
def load_cached_raw(ticker):
    """Load and validate one immutable raw parquet cache.

    The submitted grading path is deliberately offline: a missing or invalid
    cache is an explicit error rather than an implicit network request.
    """
    raw_path = DATA_DIR / f"{ticker}_raw.parquet"

    if not raw_path.exists():
        raise FileNotFoundError(
            f"Missing cached raw file: {raw_path}. "
            "The submitted notebook requires the parquet cache and does not "
            "download data during grading."
        )

    df = pd.read_parquet(raw_path)
    if df.empty:
        raise ValueError(f"Cached raw file is empty: {raw_path}")

    fields = set(
        df.columns.get_level_values(0)
        if isinstance(df.columns, pd.MultiIndex)
        else df.columns
    )
    required_fields = {
        "Open", "High", "Low", "Close", "Adj Close",
        "Volume", "Dividends", "Stock Splits",
    }
    missing_fields = required_fields - fields
    if missing_fields:
        raise ValueError(
            f"Cached raw file {raw_path.name} is missing fields: "
            f"{sorted(missing_fields)}"
        )

    dates = pd.to_datetime(df.index)
    if getattr(dates, "tz", None) is not None:
        dates = dates.tz_localize(None)
    required_end = pd.Timestamp(END_DATE) - pd.Timedelta(days=1)
    if dates.min() > pd.Timestamp(START_DATE) or dates.max() < required_end:
        raise ValueError(
            f"Cached raw file {raw_path.name} does not cover the required "
            f"window {START_DATE} to {required_end.date()}."
        )

    print(f"{ticker}: loading and validating cached raw data")
    return df


In [ ]:
raw_data = {}

for ticker in ALL_TICKERS:
    raw_data[ticker] = load_cached_raw(ticker)

## 2.1 Raw Data Profile

Before running the quality checks, the raw files are profiled to confirm their dimensions, date coverage, index structure, and available fields. This step also verifies that the cached files contain the information required for the subsequent analysis.

In [ ]:
# Summarise the basic structure of each raw dataset
profile_rows = []

for ticker, df in raw_data.items():
    profile_rows.append({
        "Ticker": ticker,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Start Date": df.index.min(),
        "End Date": df.index.max(),
        "Index Type": type(df.index).__name__
    })

raw_profile = pd.DataFrame(profile_rows)

raw_profile

In [ ]:
# Inspect the column structure returned by yfinance
print("AAPL column structure:")
print(raw_data["AAPL"].columns)

raw_data["AAPL"].head()

## 2.2 Working Data Layer

The raw yfinance files use a two-level column index, with the price field on the first level and the ticker on the second. Since each file contains only one stock, the ticker level is redundant for the subsequent analysis.

A separate working layer removes this redundant column level while leaving the cached raw parquet files unchanged. No observations or values are altered at this stage.

In [ ]:
# Create a working copy with simpler single-level column names
working_data = {}

for ticker, df in raw_data.items():
    temp = df.copy()

    if isinstance(temp.columns, pd.MultiIndex):
        temp.columns = temp.columns.get_level_values("Price")

    working_data[ticker] = temp

In [ ]:
# Verify the simplified working structure
print("Working columns:")
print(working_data["AAPL"].columns.tolist())

working_data["AAPL"].head()

In [ ]:
# Check recorded stock-split events for AMZN
amzn_splits = working_data["AMZN"].loc[
    working_data["AMZN"]["Stock Splits"] != 0,
    ["Close", "Adj Close", "Stock Splits"]
]

amzn_splits

In [ ]:
# Inspect AMZN prices around the 20-for-1 stock split
working_data["AMZN"].loc[
    "2022-06-01":"2022-06-08",
    ["Close", "Adj Close", "Stock Splits"]
]

# 3. Part A — Data Quality Report

The data-quality review applies the Lecture 2 validation workflow to all four stock files. Each check identifies potential issues before any treatment is applied. Observations remain unchanged unless the evidence supports a defensible correction.

## 3.1 Duplicate Check

Duplicate timestamps can create repeated observations and distort return calculations or downstream aggregation. The check covers both repeated trading dates and fully duplicated data rows for all four stock files.

In [ ]:
# Check duplicate trading dates and fully duplicated data rows
duplicate_results = []

for ticker, df in working_data.items():
    duplicate_date_count = df.index.duplicated(keep=False).sum()
    duplicate_row_count = df.duplicated(keep=False).sum()

    duplicate_results.append({
        "Ticker": ticker,
        "Duplicate Dates": duplicate_date_count,
        "Duplicate Rows": duplicate_row_count
    })

duplicate_summary = pd.DataFrame(duplicate_results)

display(duplicate_summary)

### Result

No duplicate trading dates or fully duplicated data rows were found across AAPL, JPM, XOM, and AMZN. No corrective action is required for this check, and all observations remain in the working dataset.

## 3.2 Trading-Calendar Gap Check

Missing dates are evaluated against the relevant U.S. exchange calendar rather than against ordinary weekdays. This distinction avoids incorrectly treating scheduled market holidays as data gaps.

AAPL and AMZN are checked against the NASDAQ calendar, while JPM and XOM are checked against the NYSE calendar. For each stock, the observed dates are compared with the expected trading sessions over the full sample window. Any expected session absent from the dataset is flagged as a missing trading day for further investigation.

In [ ]:
import pandas_market_calendars as mcal

# Map each stock to its primary exchange calendar
exchange_calendars = {
    "AAPL": "NASDAQ",
    "JPM": "NYSE",
    "XOM": "NYSE",
    "AMZN": "NASDAQ"
}

# END_DATE is exclusive in the yfinance download
analysis_end = pd.Timestamp(END_DATE) - pd.Timedelta(days=1)

calendar_gap_results = []
missing_dates_by_ticker = {}
unexpected_dates_by_ticker = {}

for ticker, df in working_data.items():
    # Load the relevant exchange trading calendar
    calendar = mcal.get_calendar(exchange_calendars[ticker])

    schedule = calendar.schedule(
        start_date=START_DATE,
        end_date=analysis_end
    )

    expected_dates = pd.DatetimeIndex(schedule.index)
    actual_dates = pd.DatetimeIndex(df.index)

    # Remove timezone information if present and compare dates only
    if expected_dates.tz is not None:
        expected_dates = expected_dates.tz_localize(None)

    if actual_dates.tz is not None:
        actual_dates = actual_dates.tz_localize(None)

    expected_dates = expected_dates.normalize()
    actual_dates = actual_dates.normalize()

    # Expected trading sessions missing from the downloaded data
    missing_dates = expected_dates.difference(actual_dates)

    # Dates in the dataset that are not valid exchange trading sessions
    unexpected_dates = actual_dates.difference(expected_dates)

    missing_dates_by_ticker[ticker] = missing_dates
    unexpected_dates_by_ticker[ticker] = unexpected_dates

    calendar_gap_results.append({
        "Ticker": ticker,
        "Exchange Calendar": exchange_calendars[ticker],
        "Expected Trading Days": len(expected_dates),
        "Observed Trading Days": len(actual_dates),
        "Missing Trading Days": len(missing_dates),
        "Unexpected Dates": len(unexpected_dates)
    })

calendar_gap_summary = pd.DataFrame(calendar_gap_results)

display(calendar_gap_summary)

# Show the actual dates only when an issue is detected
for ticker in ALL_TICKERS:
    if len(missing_dates_by_ticker[ticker]) > 0:
        print(f"\n{ticker} - Missing trading days:")
        print(missing_dates_by_ticker[ticker].strftime("%Y-%m-%d").tolist())

    if len(unexpected_dates_by_ticker[ticker]) > 0:
        print(f"\n{ticker} - Unexpected dates:")
        print(unexpected_dates_by_ticker[ticker].strftime("%Y-%m-%d").tolist())

### Result

All four stocks contain 2,513 observations, exactly matching the number of expected trading sessions in their respective exchange calendars. No missing trading days or unexpected dates were identified. Scheduled weekends and market holidays are therefore correctly excluded, and no corrective action is required.

## 3.3 Stale Price Check

A stale-price episode may indicate that a data feed has stopped updating even though trading continues. For these highly liquid equities, a run of at least three consecutive trading sessions with exactly the same `Close` is used as a screening threshold.

The rule is intended to flag suspicious observations for review rather than automatically remove them. Both the number of flagged runs and the longest identical-close run are reported for each stock.

In [ ]:
# Flag runs of at least three consecutive trading days with identical Close prices
STALE_THRESHOLD = 3

stale_summary_rows = []
stale_details = []

for ticker, df in working_data.items():
    temp = pd.DataFrame({
        "Date": df.index,
        "Close": df["Close"].to_numpy()
    })

    # Start a new run whenever the closing price changes
    temp["Run ID"] = temp["Close"].ne(temp["Close"].shift()).cumsum()

    runs = (
        temp.groupby("Run ID", as_index=False)
        .agg(
            Start_Date=("Date", "first"),
            End_Date=("Date", "last"),
            Close=("Close", "first"),
            Run_Length=("Date", "size")
        )
    )

    flagged_runs = runs[runs["Run_Length"] >= STALE_THRESHOLD].copy()

    if not flagged_runs.empty:
        flagged_runs.insert(0, "Ticker", ticker)
        stale_details.append(flagged_runs)

    stale_summary_rows.append({
        "Ticker": ticker,
        "Flagged Stale Runs": len(flagged_runs),
        "Longest Identical-Close Run": runs["Run_Length"].max()
    })

stale_summary = pd.DataFrame(stale_summary_rows)

display(stale_summary)

# Display detailed dates only if stale-price runs are found
if stale_details:
    stale_detail_table = pd.concat(stale_details, ignore_index=True)
    display(stale_detail_table)
else:
    print(f"No identical-Close runs of {STALE_THRESHOLD} or more trading days were found.")

### Result

No stale-price runs meeting the three-session screening threshold were identified for AAPL, JPM, or XOM. Their longest identical-close runs lasted only two trading sessions.

AMZN produced one potential stale-price flag: the closing price remained exactly $125.98 for three consecutive trading sessions from 26 to 28 September 2023. Since this meets the screening threshold, the underlying observations require further investigation before deciding whether any corrective action is appropriate.

### Investigation of the AMZN Stale-Price Flag

A genuinely stale price feed would typically repeat more than just the closing price. The underlying OHLCV observations are therefore examined across the flagged period.

If trading activity continued normally, variation in the intraday price range and trading volume would provide evidence that the repeated closing price was a legitimate market outcome rather than a frozen data feed.

In [ ]:
# Inspect the underlying OHLCV observations for the flagged AMZN run
amzn_stale_window = working_data["AMZN"].loc[
    "2023-09-26":"2023-09-28",
    ["Open", "High", "Low", "Close", "Volume"]
].copy()

display(amzn_stale_window)

# Check whether each field remained identical across all three sessions
stale_field_check = pd.DataFrame({
    "Field": ["Open", "High", "Low", "Close", "Volume"],
    "Unique Values": [
        amzn_stale_window[column].nunique()
        for column in ["Open", "High", "Low", "Close", "Volume"]
    ]
})

stale_field_check["Constant Across Period"] = (
    stale_field_check["Unique Values"] == 1
)

display(stale_field_check)

### Investigation Result

The three AMZN sessions share the same closing price, but their open, high, low, and trading-volume observations are not constant. The variation in both intraday prices and volume confirms that trading activity continued throughout the flagged period.

The repeated $125.98 close is therefore interpreted as a legitimate market outcome rather than a stale data feed. All three observations are retained, and no data correction is applied.

## 3.4 Return Outlier Check

Extreme daily returns may represent either genuine market events or erroneous price observations. The analysis therefore uses a staged forensic screen: a robust statistical flag is followed by volume and price-persistence cross-checks, then by cross-stock context and external verification. No rule automatically deletes an observation.

### Screening Design

**Step 1 — Return outlier detection.**  
Daily simple returns are calculated from adjusted close so that corporate actions do not create artificial price jumps. Potential outliers are identified with the modified robust z-score based on the full-sample median and median absolute deviation (MAD). An absolute robust z-score above 3.5 is used as a screening convention; it is not, by itself, evidence of a bad tick.

**Step 2 — Volume cross-check.**  
Because raw volume is strongly right-skewed, log(volume) is compared with the preceding 20 trading sessions. The event-day value is assigned an empirical rolling percentile:

- Thin volume: at or below the 20th percentile;
- Heavy volume: at or above the 80th percentile;
- otherwise: inconclusive volume support.

The 20-session window is approximately one trading month and is used consistently for all stocks. The percentile cut-offs are transparent analyst-defined operational rules, not exchange standards.

**Step 3 — Price persistence / reversal cross-check.**  
The assignment describes suspicious observations as exhibiting a “spike-and-revert” pattern. The next three trading sessions are therefore examined. If at least 50% of the event-day adjusted-price move is reversed relative to the pre-event price, the observation is classified as a material three-session reversal. The three-session horizon and 50% threshold are investigation rules rather than universal industry thresholds.

**Step 4 — Three-way triage.**

- Possible bad tick: return outlier + thin volume + material reversal.
- Likely genuine market move: return outlier + heavy volume + no material reversal.
- Needs review: every other return outlier.

This triage separates the pattern described in the assignment from persistent, heavily traded moves while preserving ambiguous cases for investigation. It is a daily-OHLCV forensic screen, not an exchange-level bad-tick adjudication rule.

**Step 5 — Cross-stock context.**  
Possible bad ticks are compared with the other sample stocks on the same date. A same-direction return outlier in another stock provides supplementary market-context evidence; its absence increases the priority of external verification. Cross-stock support is not used as a hard classification rule because the sample contains only four stocks.

**Step 6 — External verification and action.**  
Candidates are checked against linked historical price sources and market context. A candidate is removed only if independent evidence demonstrates an erroneous observation. Otherwise it remains in the cleaned data and is documented as a statistical flag.


In [ ]:
# Screen return outliers using return extremeness, volume support,
# short-horizon reversal, and cross-stock market context

RETURN_Z_THRESHOLD = 3.5

VOLUME_WINDOW = 20
THIN_VOLUME_PERCENTILE = 0.20
HEAVY_VOLUME_PERCENTILE = 0.80

REVERSAL_WINDOW = 3            # Analyst-defined investigation horizon
REVERSAL_THRESHOLD = 0.50      # Analyst-defined material-reversal threshold


# Store the complete diagnostic dataset for each stock
outlier_working = {}

for ticker, df in working_data.items():
    temp = df.copy()

    # ============================================================
    # Step 1: Return outlier detection
    # ============================================================

    temp["Return"] = temp["Adj Close"].pct_change()

    return_median = temp["Return"].median()

    return_mad = np.median(
        np.abs(
            temp["Return"].dropna()
            - return_median
        )
    )

    if return_mad == 0:
        temp["Return Robust Z"] = np.nan
    else:
        temp["Return Robust Z"] = (
            0.6745
            * (temp["Return"] - return_median)
            / return_mad
        )

    temp["Potential Outlier"] = (
        temp["Return Robust Z"].abs()
        > RETURN_Z_THRESHOLD
    )


    # ============================================================
    # Step 2: Rolling volume-percentile cross-check
    # ============================================================

    temp["Prior 20D Median Volume"] = (
        temp["Volume"].shift(1).rolling(
            window=VOLUME_WINDOW,
            min_periods=VOLUME_WINDOW
        ).median()
    )
    temp["Volume Ratio"] = (
        temp["Volume"] / temp["Prior 20D Median Volume"]
    )

    log_volume = np.log(temp["Volume"].where(temp["Volume"] > 0))
    volume_percentile = pd.Series(np.nan, index=temp.index, dtype=float)

    for i in range(VOLUME_WINDOW, len(temp)):
        prior_volume = log_volume.iloc[i - VOLUME_WINDOW:i].dropna()
        current_volume = log_volume.iloc[i]
        if len(prior_volume) < VOLUME_WINDOW or pd.isna(current_volume):
            continue
        volume_percentile.iloc[i] = (
            ((prior_volume < current_volume).sum()
             + 0.5 * (prior_volume == current_volume).sum())
            / len(prior_volume)
        )

    temp["Volume Percentile"] = volume_percentile
    temp["Thin Volume"] = temp["Volume Percentile"] <= THIN_VOLUME_PERCENTILE
    temp["Heavy Volume"] = temp["Volume Percentile"] >= HEAVY_VOLUME_PERCENTILE

    # ============================================================
    # Step 3: Short-horizon reversal
    # ============================================================

    temp["Previous Adj Close"] = (
        temp["Adj Close"].shift(1)
    )

    reversal_fraction = pd.Series(
        np.nan,
        index=temp.index,
        dtype=float
    )

    reversal_date = pd.Series(
        pd.NaT,
        index=temp.index,
        dtype="datetime64[ns]"
    )

    reversal_price = pd.Series(
        np.nan,
        index=temp.index,
        dtype=float
    )

    prices = temp["Adj Close"]

    for i in range(1, len(temp)):

        previous_price = prices.iloc[i]
        pre_event_price = prices.iloc[i - 1]

        if (
            pd.isna(previous_price)
            or pd.isna(pre_event_price)
        ):
            continue

        event_move = (
            previous_price - pre_event_price
        )

        if event_move == 0:
            reversal_fraction.iloc[i] = 0
            continue

        future_end = min(
            i + REVERSAL_WINDOW + 1,
            len(temp)
        )

        future_prices = prices.iloc[
            i + 1:future_end
        ].dropna()

        if future_prices.empty:
            continue

        # Upward event: measure subsequent movement downward
        if event_move > 0:
            retracement = (
                previous_price - future_prices
            ).clip(lower=0)

        # Downward event: measure subsequent movement upward
        else:
            retracement = (
                future_prices - previous_price
            ).clip(lower=0)

        retracement_fraction = (
            retracement / abs(event_move)
        )

        max_reversal_date = (
            retracement_fraction.idxmax()
        )

        reversal_fraction.iloc[i] = (
            retracement_fraction.loc[
                max_reversal_date
            ]
        )

        reversal_date.iloc[i] = (
            max_reversal_date
        )

        reversal_price.iloc[i] = (
            prices.loc[max_reversal_date]
        )

    temp["Max 3D Reversal Fraction"] = (
        reversal_fraction
    )

    temp["Max Reversal Date"] = (
        reversal_date
    )

    temp["Max Reversal Price"] = (
        reversal_price
    )

    temp["Material 3D Reversal"] = (
        temp["Max 3D Reversal Fraction"]
        >= REVERSAL_THRESHOLD
    )


    # ============================================================
    # Step 4: Three-way triage
    # ============================================================

    temp["Possible Bad Tick"] = (
        temp["Potential Outlier"]
        & temp["Thin Volume"]
        & temp["Material 3D Reversal"]
    )

    temp["Likely Genuine Move"] = (
        temp["Potential Outlier"]
        & temp["Heavy Volume"]
        & ~temp["Material 3D Reversal"]
    )

    temp["Needs Review"] = (
        temp["Potential Outlier"]
        & ~(
            temp["Possible Bad Tick"]
            | temp["Likely Genuine Move"]
        )
    )

    temp["Screening Category"] = np.select(
        [
            temp["Possible Bad Tick"],
            temp["Likely Genuine Move"],
        ],
        [
            "Possible bad tick",
            "Likely genuine move",
        ],
        default="Needs review"
    )


    outlier_working[ticker] = temp


# ================================================================
# Step 4: Cross-stock context
# ================================================================

candidate_rows = []

for ticker, temp in outlier_working.items():

    candidates = temp[
        temp["Possible Bad Tick"]
    ].copy()

    for date, row in candidates.iterrows():

        event_return = row["Return"]
        event_direction = np.sign(event_return)

        supporting_tickers = []

        # Check the OTHER stocks on the same trading date
        for other_ticker, other_df in outlier_working.items():

            if other_ticker == ticker:
                continue

            if date not in other_df.index:
                continue

            other_row = other_df.loc[date]

            # Cross-stock support requires:
            # 1. another robust-z outlier
            # 2. same return direction
            if (
                other_row["Potential Outlier"]
                and np.sign(other_row["Return"])
                    == event_direction
            ):
                supporting_tickers.append(
                    other_ticker
                )

        candidate_rows.append({
            "Ticker": ticker,
            "Date": date,
            "Return": row["Return"],
            "Return Robust Z": row["Return Robust Z"],
            "Volume": row["Volume"],
            "Prior 20D Median Volume":
                row["Prior 20D Median Volume"],
            "Volume Ratio":
                row["Volume Ratio"],
            "Volume Percentile":
                row["Volume Percentile"],
            "Previous Adj Close":
                row["Previous Adj Close"],
            "Adj Close":
                row["Adj Close"],
            "Max 3D Reversal Fraction":
                row["Max 3D Reversal Fraction"],
            "Max Reversal Date":
                row["Max Reversal Date"],
            "Max Reversal Price":
                row["Max Reversal Price"],
            "Screening Category": row["Screening Category"],
            "Cross-Stock Support Count":
                len(supporting_tickers),
            "Supporting Tickers":
                ", ".join(supporting_tickers)
                if supporting_tickers
                else "None"
        })


candidate_table = pd.DataFrame(
    candidate_rows
)


# ================================================================
# Summary table
# ================================================================

summary_rows = []

for ticker, temp in outlier_working.items():

    summary_rows.append({
        "Ticker": ticker,

        "Potential Outliers": int(
            temp["Potential Outlier"].sum()
        ),

        "Thin-Volume Outliers": int(
            (
                temp["Potential Outlier"]
                & temp["Thin Volume"]
            ).sum()
        ),

        "Heavy-Volume Outliers": int(
            (
                temp["Potential Outlier"]
                & temp["Heavy Volume"]
            ).sum()
        ),

        "Possible Bad Tick": int(
            temp["Possible Bad Tick"].sum()
        ),

        "Likely Genuine Move": int(
            temp["Likely Genuine Move"].sum()
        ),

        "Needs Review": int(
            temp["Needs Review"].sum()
        )
    })


outlier_summary = pd.DataFrame(
    summary_rows
)

display(outlier_summary)


print(
    "Total potential outliers:",
    outlier_summary[
        "Potential Outliers"
    ].sum()
)

print(
    "Possible bad-tick candidates:",
    outlier_summary[
        "Possible Bad Tick"
    ].sum()
)

print(
    "Likely genuine moves:",
    outlier_summary[
        "Likely Genuine Move"
    ].sum()
)

print(
    "Cases requiring review:",
    outlier_summary[
        "Needs Review"
    ].sum()
)


# ================================================================
# Candidate table with cross-stock context
# ================================================================

if not candidate_table.empty:

    # Lower support count = higher priority for external verification
    candidate_table = (
        candidate_table
        .sort_values(
            by=[
                "Cross-Stock Support Count",
                "Return Robust Z"
            ],
            key=lambda col:
                col.abs()
                if col.name == "Return Robust Z"
                else col,
            ascending=[True, False]
        )
        .reset_index(drop=True)
    )

    display(
        candidate_table.style.format({
            "Return": "{:.2%}",
            "Return Robust Z": "{:.2f}",
            "Volume": "{:,.0f}",
            "Prior 20D Median Volume":
                "{:,.0f}",
            "Volume Ratio": "{:.2f}x",
            "Volume Percentile": "{:.0%}",
            "Previous Adj Close": "${:.2f}",
            "Adj Close": "${:.2f}",
            "Max 3D Reversal Fraction":
                "{:.0%}",
            "Max Reversal Price": "${:.2f}"
        })
    )


    # Show how many candidates have 0, 1, 2, or 3 supporting stocks
    cross_stock_summary = (
        candidate_table[
            "Cross-Stock Support Count"
        ]
        .value_counts()
        .sort_index()
        .rename_axis(
            "Cross-Stock Support Count"
        )
        .reset_index(
            name="Candidate Cases"
        )
    )

    display(cross_stock_summary)


    # Highest-priority cases for external verification:
    # no same-day, same-direction robust-z support from another stock
    isolated_candidates = candidate_table[
        candidate_table[
            "Cross-Stock Support Count"
        ] == 0
    ].copy()

    print(
        "Possible bad ticks without same-day cross-stock support:",
        len(isolated_candidates)
    )

    if not isolated_candidates.empty:
        display(
            isolated_candidates.style.format({
                "Return": "{:.2%}",
                "Return Robust Z": "{:.2f}",
                "Volume": "{:,.0f}",
                "Prior 20D Median Volume":
                    "{:,.0f}",
                "Volume Ratio": "{:.2f}x",
                "Volume Percentile": "{:.0%}",
                "Previous Adj Close": "${:.2f}",
                "Adj Close": "${:.2f}",
                "Max 3D Reversal Fraction":
                    "{:.0%}",
                "Max Reversal Price": "${:.2f}"
            })
        )

else:
    print(
        "No possible bad-tick candidates "
        "were identified."
    )

### External Verification of Possible Bad-Tick Candidates and Sources

The refined triage identifies six possible bad-tick candidates from the 257 statistical return outliers. Four have same-direction support from another stock in the sample (AAPL/JPM on 26 March and 1 April 2020); two XOM candidates are isolated within this four-stock universe. Cross-stock support is treated as contextual evidence only.

The linked historical-price pages provide an auditable source for checking the return direction and magnitude. Market context is used as supporting evidence rather than as a computational dependency. No observation is automatically deleted: all six candidates remain in the cleaned dataset unless independent evidence demonstrates an erroneous print.

**Reproducible price-verification sources (accessed 5 September 2026):**

- [AAPL historical data — Yahoo Finance](https://finance.yahoo.com/quote/AAPL/history/)
- [JPM historical data — Yahoo Finance](https://finance.yahoo.com/quote/JPM/history/)
- [XOM historical data — Yahoo Finance](https://finance.yahoo.com/quote/XOM/history/)
- [AMZN historical data — Yahoo Finance](https://finance.yahoo.com/quote/AMZN/history/)

The framework is a daily-OHLCV forensic screen, not an exchange-level trade-cancellation rule.


#### Verification Evidence Table

The table below lists the six possible bad-tick candidates produced by the refined screen. The external-return column records the linked historical-price comparison used for manual verification; rounding and adjustment conventions may create immaterial differences.

| Candidate | yfinance Return | External Return | Market Context | Price source | Action |
|---|---:|---:|---|---|---|
| AAPL — 2020-03-26 | +5.26% | ≈ +5.26% | Broad COVID-19 crisis rebound and policy-support expectations. | [Yahoo Finance history](https://finance.yahoo.com/quote/AAPL/history/) | Keep |
| AAPL — 2020-04-01 | -5.26% | ≈ -5.26% | Renewed COVID-19 and economic uncertainty drove a broad risk-off session. | [Yahoo Finance history](https://finance.yahoo.com/quote/AAPL/history/) | Keep |
| JPM — 2020-03-26 | +6.97% | ≈ +6.97% | Broad U.S. equity rebound during the COVID-19 crisis. | [Yahoo Finance history](https://finance.yahoo.com/quote/JPM/history/) | Keep |
| JPM — 2020-04-01 | -6.30% | ≈ -6.30% | Continued COVID-19 stress and credit-market uncertainty. | [Yahoo Finance history](https://finance.yahoo.com/quote/JPM/history/) | Keep |
| XOM — 2020-04-08 | +6.33% | ≈ +6.32% | U.S. equities rallied on improving COVID-19 expectations; energy stocks were strong. | [Yahoo Finance history](https://finance.yahoo.com/quote/XOM/history/) | Keep |
| XOM — 2020-04-29 | +5.54% | ≈ +5.53% | Reopening expectations improved risk sentiment while crude oil rebounded. | [Yahoo Finance history](https://finance.yahoo.com/quote/XOM/history/) | Keep |

### Result

The refined screen produces six review candidates rather than 32 broad high-priority flags. The linked price histories and market context support retaining these observations; no confirmed bad tick is identified. The candidates remain documented as statistical flags rather than being removed from the cleaned dataset.


## 3.5 Zero or Negative Volume Check

Zero or negative trading volume on a valid trading session may indicate a data-quality problem. Each stock is therefore checked for observations where reported `Volume` is less than or equal to zero.

Any such observation would require further investigation before use in downstream analysis. No value is modified or removed automatically.

In [ ]:
# Check for zero or negative trading volume
volume_issue_rows = []
volume_issue_details = []

for ticker, df in working_data.items():

    invalid_volume = df["Volume"] <= 0

    volume_issue_rows.append({
        "Ticker": ticker,
        "Zero Volume Days": int((df["Volume"] == 0).sum()),
        "Negative Volume Days": int((df["Volume"] < 0).sum()),
        "Total Invalid Volume Days": int(invalid_volume.sum())
    })

    if invalid_volume.any():
        flagged = df.loc[
            invalid_volume,
            ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
        ].copy()

        flagged.insert(0, "Ticker", ticker)
        flagged.insert(1, "Date", flagged.index)
        flagged.reset_index(drop=True, inplace=True)

        volume_issue_details.append(flagged)

volume_issue_summary = pd.DataFrame(volume_issue_rows)

display(volume_issue_summary)

if volume_issue_details:
    volume_issue_detail_table = pd.concat(
        volume_issue_details,
        ignore_index=True
    )

    display(volume_issue_detail_table)

else:
    print("No zero or negative volume observations were found.")

### Result

No zero or negative volume observations were identified for AAPL, JPM, XOM, or AMZN. All reported trading-session volumes are strictly positive, so no corrective action is required for this check.

## 3.6 Corporate Action: Raw vs Adjusted Price

Amazon completed a 20-for-1 stock split on 6 June 2022. A stock split changes the number of shares outstanding and the nominal price per share, but it does not by itself represent an equivalent economic gain or loss for investors.

Yahoo Finance reports historical `Close` prices on a split-adjusted basis. Consequently, the downloaded AMZN `Close` series remains continuous across the split rather than displaying the large nominal price discontinuity illustrated in Lecture 2. `Adj Close` is also continuous and incorporates additional adjustments where applicable.

To reproduce the economically unadjusted nominal price series required for the split analysis, the recorded `Stock Splits` factor is reversed. Pre-split observations are multiplied by the cumulative split factor applicable after each date, while the original cached raw files and the main working dataset remain unchanged.

The reconstructed pre-split close on 3 June 2022 is approximately $2,447, which is consistent with independently reported historical market prices. This external check supports the validity of the reconstruction rather than introducing an alternative computational data source.

The comparison below quantifies the return error that would result from treating the mechanical split-induced price change as an ordinary investment return.

In [ ]:
# Analyse AMZN's 20-for-1 stock split without modifying the working dataset
amzn_corp = working_data["AMZN"].copy()

# Convert non-split days to a neutral factor of 1
split_ratio = amzn_corp["Stock Splits"].replace(0, 1)

# Reverse Yahoo Finance's historical split adjustment
# Only splits occurring after a given date affect that historical price
future_split_factor = (
    split_ratio.iloc[::-1]
    .cumprod()
    .iloc[::-1]
    / split_ratio
)

amzn_corp["Reconstructed Nominal Close"] = (
    amzn_corp["Close"] * future_split_factor
)

# Identify the split date
split_dates = amzn_corp.index[
    amzn_corp["Stock Splits"] > 0
]

if len(split_dates) == 0:
    raise ValueError(
        "No AMZN stock split was found in the sample period."
    )

split_date = split_dates[0]
split_position = amzn_corp.index.get_loc(split_date)
pre_split_date = amzn_corp.index[split_position - 1]

split_factor = amzn_corp.loc[
    split_date,
    "Stock Splits"
]

# Extract prices immediately before and after the split
pre_nominal_close = amzn_corp.loc[
    pre_split_date,
    "Reconstructed Nominal Close"
]

post_nominal_close = amzn_corp.loc[
    split_date,
    "Reconstructed Nominal Close"
]

pre_adjusted_close = amzn_corp.loc[
    pre_split_date,
    "Adj Close"
]

post_adjusted_close = amzn_corp.loc[
    split_date,
    "Adj Close"
]

# Return that would be obtained from the unadjusted nominal-price series
naive_nominal_return = (
    post_nominal_close / pre_nominal_close - 1
)

# Economically meaningful return after corporate-action adjustment
adjusted_return = (
    post_adjusted_close / pre_adjusted_close - 1
)

return_error_pp = abs(
    naive_nominal_return - adjusted_return
) * 100

# Show observations around the split
split_window = amzn_corp.loc[
    split_date - pd.Timedelta(days=7):
    split_date + pd.Timedelta(days=7),
    [
        "Close",
        "Adj Close",
        "Stock Splits",
        "Reconstructed Nominal Close"
    ]
]

display(split_window)

print(f"Pre-split trading day: {pre_split_date.date()}")
print(f"Split date: {split_date.date()}")
print(f"Split ratio: {split_factor:.0f}:1")
print(
    f"Reconstructed pre-split nominal close: "
    f"${pre_nominal_close:,.2f}"
)
print(
    f"Post-split nominal close: "
    f"${post_nominal_close:,.2f}"
)
print(
    f"Naive nominal-price return: "
    f"{naive_nominal_return:.2%}"
)
print(
    f"Adjusted return: "
    f"{adjusted_return:.2%}"
)
print(
    f"Return error: "
    f"{return_error_pp:.2f} percentage points"
)

# Plot the nominal and adjusted series around the split
plot_window = amzn_corp.loc[
    split_date - pd.Timedelta(days=30):
    split_date + pd.Timedelta(days=30)
]

plt.figure(figsize=(10, 5))

plt.plot(
    plot_window.index,
    plot_window["Reconstructed Nominal Close"],
    label="Reconstructed Nominal Close"
)

plt.plot(
    plot_window.index,
    plot_window["Adj Close"],
    label="Adjusted Close"
)

plt.axvline(
    split_date,
    linestyle="--",
    label="20-for-1 Split"
)

plt.title(
    "AMZN Price Series Around the 20-for-1 Stock Split"
)
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Result

AMZN's recorded 20-for-1 split implies a reconstructed nominal closing price of approximately $2,447 on 3 June 2022, compared with $124.79 on the first post-split trading session. Treating this mechanical price change as an ordinary return would incorrectly imply a one-day loss of approximately 94.90%.

The split-adjusted series instead produces a return of approximately +1.99%. Using the unadjusted nominal series would therefore introduce an error of approximately 96.89 percentage points.

The corporate action is a mechanical change in price scale rather than an economic loss. The adjusted series is therefore retained for return-based downstream analysis, while the reconstructed nominal series is used only to demonstrate the effect of the stock split.

## 3.7 Data Quality Summary and Cleaned Dataset

The six data-quality checks are summarised below. Potential issues were investigated before any treatment was applied. Since no confirmed erroneous observations were identified, no price or volume records are removed or modified.

The cleaned dataset therefore preserves all original observations while retaining the simplified single-level column structure created in the working layer.

In [ ]:
# Summarise the six Part A data-quality checks
data_quality_summary = pd.DataFrame([
    {
        "Check": "Duplicates",
        "What Was Found": "0 duplicate dates; 0 duplicate rows across all four stocks",
        "Action": "Keep",
        "Justification": "No duplicated observations were identified."
    },
    {
        "Check": "Trading-calendar gaps",
        "What Was Found": "0 missing trading days; 0 unexpected dates; 2,513 valid sessions per stock",
        "Action": "Keep",
        "Justification": "Observed dates exactly match the relevant NYSE or NASDAQ trading calendars."
    },
    {
        "Check": "Stale prices",
        "What Was Found": "1 flagged run: AMZN, 26–28 Sep 2023, Close = $125.98 for 3 sessions",
        "Action": "Keep",
        "Justification": "Open, high, low, and volume varied across the three sessions, confirming active trading rather than a stale feed."
    },
    {
        "Check": "Return outliers",
        "What Was Found": "257 robust-z flags; 6 possible bad-tick candidates; 139 likely genuine moves; 112 needs-review cases; 0 confirmed bad ticks",
        "Action": "Keep; flag where applicable",
        "Justification": "Volume/reversal flags are retained for review; linked price histories and market context support keeping the observations, and no confirmed bad tick is identified."
    },
    {
        "Check": "Zero / negative volume",
        "What Was Found": "0 zero-volume days; 0 negative-volume days across all four stocks",
        "Action": "Keep",
        "Justification": "All reported trading-session volumes are strictly positive."
    },
    {
        "Check": "Adjusted vs raw close",
        "What Was Found": "AMZN 20-for-1 split on 6 Jun 2022; naive return = -94.90%; adjusted return = +1.99%",
        "Action": "Keep",
        "Justification": "The nominal price discontinuity is caused by the split; adjusted prices avoid a -96.89 percentage-point return error."
    }
])

display(data_quality_summary)

# Create the cleaned layer for downstream analysis
# No observations are removed because no confirmed data errors were found
cleaned_data = {
    ticker: df.copy()
    for ticker, df in working_data.items()
}

print("Cleaned datasets created:")
for ticker, df in cleaned_data.items():
    print(f"{ticker}: {len(df)} observations")

# 4. Part B — Benford Forensic Screen

Benford's Law provides a first-pass forensic screen for numerical datasets whose values arise naturally across a sufficiently broad range of magnitudes.

Under Benford's Law, the probability that the first significant digit equals \(d\) is:

$$
P(D=d)=\log_{10}\left(1+\frac{1}{d}\right),
\qquad d=1,\ldots,9
$$

This implies that smaller leading digits occur more frequently. For example, the theoretical probability of a leading digit of 1 is approximately 30.1%, while the probability of a leading digit of 9 is approximately 4.6%.

The analysis uses the three main stocks: AAPL, JPM, and XOM. For each stock, three daily series are examined:

- trading volume;
- value traded; and
- absolute daily price change.

## 4.1 Construction of Benford Test Series

Daily value traded is approximated as:

$$
\text{Value Traded}_t
=
\text{Close}_t
\times
\text{Volume}_t
$$

Absolute daily price change is calculated as:

$$
\left|\Delta P_t\right|
=
\left|
\text{Adj Close}_t
-
\text{Adj Close}_{t-1}
\right|
$$

Adjusted prices are used for the price-change series so that corporate actions do not create mechanical price movements.

Benford analysis applies only to positive, non-zero values because zero has no leading significant digit. Missing observations and zero absolute price changes are therefore excluded before extracting the first digit.

In [ ]:
# Construct the three Benford test series for the main stocks

benford_data = {}

for ticker in MAIN_TICKERS:
    df = cleaned_data[ticker].copy()

    # Daily trading volume
    df["Benford Volume"] = df["Volume"]

    # Daily value traded: closing price multiplied by trading volume
    df["Benford Value Traded"] = (
        df["Close"] * df["Volume"]
    )

    # Absolute daily price change using adjusted prices
    df["Benford Abs Price Change"] = (
        df["Adj Close"].diff().abs()
    )

    benford_data[ticker] = df


# Extract the first significant digit from positive numerical values
def first_digit(series):
    values = series.dropna()
    values = values[values > 0]

    digits = np.floor(
        values / (
            10 ** np.floor(np.log10(values))
        )
    ).astype(int)

    return digits


# Store leading digits for each stock and each test series
benford_digits = {}

series_columns = {
    "Volume": "Benford Volume",
    "Value Traded": "Benford Value Traded",
    "Absolute Price Change": "Benford Abs Price Change"
}

sample_summary_rows = []

for ticker in MAIN_TICKERS:
    benford_digits[ticker] = {}

    for series_name, column_name in series_columns.items():

        digits = first_digit(
            benford_data[ticker][column_name]
        )

        benford_digits[ticker][series_name] = digits

        sample_summary_rows.append({
            "Ticker": ticker,
            "Series": series_name,
            "Valid Observations": len(digits),
            "Minimum Value": benford_data[ticker][column_name]
                .replace(0, np.nan)
                .min(),
            "Maximum Value": benford_data[ticker][column_name]
                .max()
        })


benford_sample_summary = pd.DataFrame(
    sample_summary_rows
)

display(
    benford_sample_summary.style.format({
        "Minimum Value": "{:,.4f}",
        "Maximum Value": "{:,.4f}"
    })
)


# Preview the first few extracted digits
for ticker in MAIN_TICKERS:
    print(f"\n{ticker}")

    for series_name in series_columns:
        print(
            f"{series_name}:",
            benford_digits[ticker][series_name]
            .head(10)
            .tolist()
        )

## 4.2 Benford Distributions and Chi-Square Tests

### 4.2.1 Individual-Stock Tests

The observed leading-digit distributions are compared with the theoretical Benford distribution for each stock and each numerical series.

For leading digit \(d \in \{1,\ldots,9\}\), the expected probability is:

$$
P(D=d)=\log_{10}\left(1+\frac{1}{d}\right)
$$

Goodness of fit is evaluated using the chi-square statistic:

$$
\chi^2
=
\sum_{d=1}^{9}
\frac{(O_d-E_d)^2}{E_d}
$$

where \(O_d\) and \(E_d\) denote the observed and expected counts for digit \(d\).

The null hypothesis is that the observed leading-digit distribution follows Benford's Law. A p-value below 0.05 indicates statistically significant deviation and is treated as a forensic flag for further interpretation rather than evidence of manipulation by itself.

In [ ]:
# Compare observed leading-digit distributions with Benford's Law
digits = np.arange(1, 10)

benford_probabilities = np.log10(
    1 + 1 / digits
)

chi_square_results = []
digit_distribution_rows = []

for ticker in MAIN_TICKERS:

    for series_name in series_columns:

        observed_digits = (
            benford_digits[ticker][series_name]
        )

        # Observed counts for digits 1 to 9
        observed_counts = (
            observed_digits
            .value_counts()
            .reindex(digits, fill_value=0)
            .sort_index()
        )

        sample_size = observed_counts.sum()

        # Expected Benford counts
        expected_counts = (
            benford_probabilities
            * sample_size
        )

        # Chi-square goodness-of-fit test
        chi2_stat, p_value = chisquare(
            f_obs=observed_counts.values,
            f_exp=expected_counts
        )

        chi_square_results.append({
            "Ticker": ticker,
            "Series": series_name,
            "N": sample_size,
            "Chi-Square": chi2_stat,
            "p-value": p_value,
            "Benford at 5%": (
                "Fail to reject"
                if p_value >= 0.05
                else "Reject"
            )
        })

        # Store observed proportions for plotting
        observed_proportions = (
            observed_counts / sample_size
        )

        for digit in digits:
            digit_distribution_rows.append({
                "Ticker": ticker,
                "Series": series_name,
                "Digit": digit,
                "Observed Proportion":
                    observed_proportions.loc[digit],
                "Benford Proportion":
                    benford_probabilities[digit - 1]
            })


chi_square_summary = pd.DataFrame(
    chi_square_results
)

digit_distribution = pd.DataFrame(
    digit_distribution_rows
)


# Display chi-square results
display(
    chi_square_summary.style.format({
        "Chi-Square": "{:.2f}",
        "p-value": "{:.4f}"
    })
)


# Plot observed distributions against Benford's Law
for series_name in series_columns:

    plt.figure(figsize=(9, 5))

    plt.plot(
        digits,
        benford_probabilities,
        marker="o",
        linewidth=2,
        label="Benford"
    )

    for ticker in MAIN_TICKERS:

        plot_data = digit_distribution[
            (digit_distribution["Ticker"] == ticker)
            & (digit_distribution["Series"] == series_name)
        ]

        plt.plot(
            plot_data["Digit"],
            plot_data["Observed Proportion"],
            marker="o",
            label=ticker
        )

    plt.title(
        f"Leading-Digit Distribution — {series_name}"
    )
    plt.xlabel("Leading Digit")
    plt.ylabel("Proportion")
    plt.xticks(digits)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

### Result

The individual Benford tests show substantial differences across the three numerical series.

Trading volume strongly rejects Benford's Law for all three stocks. The observed distributions are highly concentrated in a relatively narrow numerical range, producing large excesses of particular leading digits. For example, JPM and XOM exhibit substantially more leading-digit 1 observations than the Benford expectation of approximately 30.1%. This result is consistent with the limited order-of-magnitude range of daily trading volume and does not by itself indicate anomalous or manipulated data.

Value traded also rejects Benford's Law for AAPL, JPM, and XOM. Although multiplying price by volume expands the numerical range, each individual stock's daily value traded remains relatively bounded rather than spanning many orders of magnitude. The strong rejection is therefore interpreted primarily as a limitation of Benford applicability for these stock-level series.

Absolute daily price change produces a much closer visual fit to the theoretical Benford distribution. This series spans a substantially wider range of magnitudes than either volume or value traded. XOM fails to reject Benford at the 5% significance level (\(p=0.2865\)), while AAPL (\(p=0.0151\)) and JPM (\(p=0.0119\)) show statistically significant but visually modest deviations.

Overall, the results demonstrate that rejection of Benford's Law must be interpreted in light of the underlying data-generating process and numerical range. A statistically significant Benford deviation is treated as a forensic flag rather than evidence of manipulation.

### 4.2.2 Pooled Tests

The three stocks are pooled within each numerical series and the Benford goodness-of-fit test is repeated.

Pooling provides a comparison with the individual-stock results and helps determine whether the observed deviations are stock-specific or structural features of the underlying series.

In [ ]:
# Pool the three stocks within each Benford test series

pooled_results = []
pooled_distribution_rows = []
pooled_digits = {}

for series_name in series_columns:

    # Combine leading digits from AAPL, JPM, and XOM
    combined_digits = pd.concat(
        [
            benford_digits[ticker][series_name]
            for ticker in MAIN_TICKERS
        ],
        ignore_index=True
    )

    pooled_digits[series_name] = combined_digits

    # Observed leading-digit counts
    observed_counts = (
        combined_digits
        .value_counts()
        .reindex(digits, fill_value=0)
        .sort_index()
    )

    sample_size = observed_counts.sum()

    # Expected counts under Benford's Law
    expected_counts = (
        benford_probabilities * sample_size
    )

    # Chi-square goodness-of-fit test
    chi2_stat, p_value = chisquare(
        f_obs=observed_counts.values,
        f_exp=expected_counts
    )

    # Count how many of the three individual-stock tests rejected Benford
    individual_rejections = (
        chi_square_summary[
            chi_square_summary["Series"] == series_name
        ]["Benford at 5%"]
        .eq("Reject")
        .sum()
    )

    pooled_results.append({
        "Series": series_name,
        "Pooled N": sample_size,
        "Chi-Square": chi2_stat,
        "p-value": p_value,
        "Benford at 5%": (
            "Fail to reject"
            if p_value >= 0.05
            else "Reject"
        ),
        "Individual Tests Rejected": f"{individual_rejections}/3"
    })

    # Store pooled observed proportions for plotting
    observed_proportions = (
        observed_counts / sample_size
    )

    for digit in digits:
        pooled_distribution_rows.append({
            "Series": series_name,
            "Digit": digit,
            "Observed Proportion":
                observed_proportions.loc[digit],
            "Benford Proportion":
                benford_probabilities[digit - 1]
        })


pooled_benford_summary = pd.DataFrame(
    pooled_results
)

pooled_distribution = pd.DataFrame(
    pooled_distribution_rows
)

# Display pooled test results
display(
    pooled_benford_summary.style.format({
        "Chi-Square": "{:.2f}",
        "p-value": "{:.3e}"
    })
)


# Plot each pooled distribution against Benford's Law
for series_name in series_columns:

    plot_data = pooled_distribution[
        pooled_distribution["Series"] == series_name
    ]

    plt.figure(figsize=(8, 5))

    plt.plot(
        digits,
        benford_probabilities,
        marker="o",
        linewidth=2,
        label="Benford"
    )

    plt.plot(
        plot_data["Digit"],
        plot_data["Observed Proportion"],
        marker="o",
        linewidth=2,
        label="Pooled Observed"
    )

    plt.title(
        f"Pooled Leading-Digit Distribution — {series_name}"
    )
    plt.xlabel("Leading Digit")
    plt.ylabel("Proportion")
    plt.xticks(digits)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4.3 Interpretation

The results show that Benford conformity depends strongly on the numerical characteristics of the underlying series rather than sample size alone.

### Trading Volume

Daily trading volume does not conform to Benford's Law. AAPL, JPM, and XOM each reject the Benford null individually, and the pooled series also strongly rejects it (\(\chi^2=1818.43\), \(p<0.001\)).

The pooled distribution shows a particularly large excess of leading digit 1 relative to the Benford expectation of approximately 30.1%. This deviation is structural rather than evidence of manipulation. Daily trading volume for these highly liquid stocks is concentrated within a relatively narrow economic range, typically from several million to several tens of millions of shares, rather than spanning many orders of magnitude. Pooling the stocks increases the sample size but does not remove this underlying range restriction.

### Value Traded

Value traded also fails to conform to Benford's Law. All three individual-stock tests reject the null, while the pooled test produces \(\chi^2=2106.59\) with \(p<0.001\).

Although multiplying price by volume broadens the numerical range, daily value traded for each large-cap stock remains relatively bounded. The pooled distribution therefore retains a substantial excess of leading digit 1 and differs markedly from the theoretical Benford distribution. The rejection is interpreted as a consequence of the data-generating process rather than a forensic indication of manipulation.

### Absolute Daily Price Change

Absolute daily price change provides substantially stronger Benford conformity. XOM fails to reject Benford individually (\(p=0.2865\)), while AAPL (\(p=0.0151\)) and JPM (\(p=0.0119\)) show statistically significant but visually modest deviations.

After pooling the three stocks, the observed distribution closely tracks the theoretical Benford curve. The pooled chi-square statistic is 14.93 with \(p=0.0606\), so the Benford null is not rejected at the 5% significance level.

This stronger fit is consistent with the much wider numerical range of absolute price changes, which extends from very small fractions of a dollar to large multi-dollar movements and therefore spans several orders of magnitude. Pooling also combines stock-specific deviations that partially offset one another, producing an aggregate distribution closer to the Benford benchmark.

### Overall Interpretation

The evidence indicates that absolute daily price change is the most appropriate of the three series for a Benford-style forensic screen in this sample. Volume and value traded exhibit strong structural departures from Benford because their observations are concentrated within comparatively narrow ranges.

The results also illustrate an important limitation of Benford analysis: statistical rejection should not be interpreted as evidence of manipulation without considering whether the underlying series satisfies the conditions under which Benford's Law is expected to apply. Benford deviations are therefore treated as forensic signals requiring interpretation rather than proof of anomalous behaviour.

## 4.4 Detection Experiment

The final experiment evaluates whether a Benford-based screen can detect deliberate contamination in the pooled value-traded series.

The clean pooled value-traded series already rejects Benford's Law, so a contaminated series cannot be classified as detected merely because its p-value is below 0.05. Instead, the experiment evaluates whether fabrication causes the chi-square statistic to move outside the range normally produced by adding the same number of genuine observations.

To mimic human fabrication, synthetic observations are generated from plausible historical value-traded magnitudes and then compressed toward the sample median. Specifically, the log of each sampled historical value is moved halfway toward the log-median, with a small amount of random noise added so that fabricated observations are not exact copies of genuine values. This produces invented observations that remain economically plausible but are more concentrated in a "comfortable" central region.

A compression parameter of $\lambda=0.50$ is used. This analyst-defined setting was selected after a pilot sensitivity check to represent a moderate degree of clustering rather than an extreme fabrication pattern.

For each contamination level of 50, 100, and 200 rows:

1. a same-size genuine-data bootstrap distribution is generated;
2. its 95% chi-square interval is estimated from 1,000 simulations;
3. fabricated observations are injected into the clean pooled series;
4. the Benford chi-square statistic is recalculated; and
5. fabrication is classified as detected when the contaminated statistic falls outside the corresponding 95% genuine-data control interval.

A contamination level is considered **reliably detected** when at least 95% of the 1,000 fabricated simulations are detected. The simulation count, control interval, reliability threshold, and fabrication parameter are analyst-defined experimental choices.

In [ ]:
# ------------------------------------------------------------
# Part B.4 — Benford Detection Experiment
# ------------------------------------------------------------

FABRICATION_SIZES = [50, 100, 200]

FABRICATION_LAMBDA = 0.50

N_SIMULATIONS = 1000

CONTROL_INTERVAL = 0.95
RELIABILITY_THRESHOLD = 0.95

JITTER_SIGMA = 0.01

RANDOM_SEED = 5009

rng = np.random.default_rng(
    RANDOM_SEED
)


# ------------------------------------------------------------
# 1. Build the clean pooled Value Traded series
# ------------------------------------------------------------

pooled_value_traded = pd.concat(
    [
        benford_data[ticker][
            "Benford Value Traded"
        ].dropna()
        for ticker in MAIN_TICKERS
    ],
    ignore_index=True
)

pooled_value_traded = (
    pooled_value_traded[
        pooled_value_traded > 0
    ]
    .reset_index(drop=True)
)

clean_values = (
    pooled_value_traded.to_numpy()
)

median_value = np.median(
    clean_values
)

log_median = np.log(
    median_value
)


# ------------------------------------------------------------
# 2. Benford test helper
# ------------------------------------------------------------

def run_benford_test(values):

    values = pd.Series(
        values
    ).dropna()

    values = values[
        values > 0
    ]

    leading_digits = first_digit(
        values
    )

    observed_counts = (
        leading_digits
        .value_counts()
        .reindex(
            digits,
            fill_value=0
        )
        .sort_index()
    )

    n = observed_counts.sum()

    expected_counts = (
        benford_probabilities
        * n
    )

    chi2_stat, p_value = chisquare(
        f_obs=observed_counts.values,
        f_exp=expected_counts
    )

    proportions = (
        observed_counts / n
    )

    return {
        "N": n,
        "Chi-Square": chi2_stat,
        "p-value": p_value,
        "Proportions": proportions
    }


# ------------------------------------------------------------
# 3. Clean baseline
# ------------------------------------------------------------

baseline_test = run_benford_test(
    clean_values
)

baseline_chi2 = (
    baseline_test["Chi-Square"]
)

print(
    f"Clean pooled observations: "
    f"{len(clean_values)}"
)

print(
    f"Median Value Traded: "
    f"${median_value:,.0f}"
)

print(
    f"Fabrication lambda: "
    f"{FABRICATION_LAMBDA:.2f}"
)

print(
    f"Baseline chi-square: "
    f"{baseline_chi2:.2f}"
)

print(
    f"Baseline p-value: "
    f"{baseline_test['p-value']:.3e}"
)


# ------------------------------------------------------------
# 4. Control interval settings
# ------------------------------------------------------------

alpha = (
    1 - CONTROL_INTERVAL
)

lower_percentile = (
    100 * alpha / 2
)

upper_percentile = (
    100 * (1 - alpha / 2)
)


# ------------------------------------------------------------
# 5. Run the experiment
# ------------------------------------------------------------

detection_results = []

representative_tests = {}


for n_fake in FABRICATION_SIZES:

    # ========================================================
    # 5.1 Genuine-data bootstrap control
    # ========================================================

    control_chi2 = []

    for _ in range(
        N_SIMULATIONS
    ):

        genuine_additions = rng.choice(
            clean_values,
            size=n_fake,
            replace=True
        )

        control_values = np.concatenate(
            [
                clean_values,
                genuine_additions
            ]
        )

        control_test = run_benford_test(
            control_values
        )

        control_chi2.append(
            control_test[
                "Chi-Square"
            ]
        )


    control_chi2 = np.array(
        control_chi2
    )

    control_lower = np.percentile(
        control_chi2,
        lower_percentile
    )

    control_upper = np.percentile(
        control_chi2,
        upper_percentile
    )


    # ========================================================
    # 5.2 Fabricated observations
    # ========================================================

    fabricated_chi2 = []

    representative_values = None

    for simulation in range(
        N_SIMULATIONS
    ):

        # Sample economically plausible historical magnitudes.
        reference_values = rng.choice(
            clean_values,
            size=n_fake,
            replace=True
        )

        reference_logs = np.log(
            reference_values
        )


        # Compress fabricated values toward the typical
        # value-traded level.
        fabricated_logs = (
            (
                1
                - FABRICATION_LAMBDA
            )
            * reference_logs
            +
            FABRICATION_LAMBDA
            * log_median
        )


        # Add small noise so the observations are invented
        # rather than exact transformations of historical rows.
        fabricated_logs += rng.normal(
            loc=0.0,
            scale=JITTER_SIGMA,
            size=n_fake
        )


        fabricated_values = np.exp(
            fabricated_logs
        )


        contaminated_values = np.concatenate(
            [
                clean_values,
                fabricated_values
            ]
        )


        contaminated_test = run_benford_test(
            contaminated_values
        )

        fabricated_chi2.append(
            contaminated_test[
                "Chi-Square"
            ]
        )


        # Store one example for the before/after plot.
        if simulation == 0:

            representative_values = (
                contaminated_values
            )


    fabricated_chi2 = np.array(
        fabricated_chi2
    )


    # ========================================================
    # 5.3 Two-sided detection
    # ========================================================

    detected = (
        (
            fabricated_chi2
            < control_lower
        )
        |
        (
            fabricated_chi2
            > control_upper
        )
    )

    detection_rate = np.mean(
        detected
    )


    # ========================================================
    # 5.4 Representative distribution
    # ========================================================

    representative_tests[
        n_fake
    ] = run_benford_test(
        representative_values
    )


    # ========================================================
    # 5.5 Store results
    # ========================================================

    detection_results.append({

        "Fabricated Rows":
            n_fake,

        "Contamination Rate":
            (
                n_fake
                /
                (
                    len(clean_values)
                    + n_fake
                )
            ),

        "Baseline Chi-Square":
            baseline_chi2,

        "Control 95% Lower":
            control_lower,

        "Control 95% Upper":
            control_upper,

        "Median Control Chi-Square":
            np.median(
                control_chi2
            ),

        "Median Fabricated Chi-Square":
            np.median(
                fabricated_chi2
            ),

        "Detection Rate":
            detection_rate,

        "Reliably Detected":
            (
                "Yes"
                if detection_rate
                >= RELIABILITY_THRESHOLD
                else "No"
            )
    })


# ------------------------------------------------------------
# 6. Summary table
# ------------------------------------------------------------

detection_summary = pd.DataFrame(
    detection_results
)

display(
    detection_summary.style.format({

        "Contamination Rate":
            "{:.2%}",

        "Baseline Chi-Square":
            "{:.2f}",

        "Control 95% Lower":
            "{:.2f}",

        "Control 95% Upper":
            "{:.2f}",

        "Median Control Chi-Square":
            "{:.2f}",

        "Median Fabricated Chi-Square":
            "{:.2f}",

        "Detection Rate":
            "{:.1%}"
    })
)


# ------------------------------------------------------------
# 7. Before / after leading-digit distributions
# ------------------------------------------------------------

for n_fake in FABRICATION_SIZES:

    contaminated_test = (
        representative_tests[
            n_fake
        ]
    )

    plt.figure(
        figsize=(8, 5)
    )

    plt.plot(
        digits,
        benford_probabilities,
        marker="o",
        label="Benford"
    )

    plt.plot(
        digits,
        baseline_test[
            "Proportions"
        ].values,
        marker="o",
        label="Clean"
    )

    plt.plot(
        digits,
        contaminated_test[
            "Proportions"
        ].values,
        marker="o",
        label=(
            f"Clean + "
            f"{n_fake} fabricated rows"
        )
    )

    plt.title(
        "Benford Detection Experiment — "
        f"{n_fake} Fabricated Rows"
    )

    plt.xlabel(
        "Leading Digit"
    )

    plt.ylabel(
        "Proportion"
    )

    plt.xticks(
        digits
    )

    plt.legend()

    plt.grid(
        alpha=0.3
    )

    plt.tight_layout()

    plt.show()


# ------------------------------------------------------------
# 8. Smallest reliably detected contamination
# ------------------------------------------------------------

reliably_detected = (
    detection_summary[
        detection_summary[
            "Reliably Detected"
        ]
        == "Yes"
    ]
)


if not reliably_detected.empty:

    smallest_detected = (
        reliably_detected[
            "Fabricated Rows"
        ].min()
    )

    smallest_row = (
        detection_summary[
            detection_summary[
                "Fabricated Rows"
            ]
            == smallest_detected
        ]
        .iloc[0]
    )

    print(
        f"Smallest reliably detected contamination: "
        f"{smallest_detected} fabricated rows"
    )

    print(
        f"Contamination rate: "
        f"{smallest_row['Contamination Rate']:.2%}"
    )

    print(
        f"Detection rate: "
        f"{smallest_row['Detection Rate']:.1%}"
    )

else:

    print(
        "None of the tested contamination levels "
        "was reliably detected under the "
        "95% reliability criterion."
    )

### Result

The detection experiment identifies a clear threshold in the ability of the Benford-based screen to distinguish fabricated observations from normal sampling variation.

With 50 fabricated rows, corresponding to approximately 0.66% contamination, the detection rate is 47.1%. This is well below the 95% reliability criterion, so this contamination level is not reliably detected.

Increasing the contamination to 100 fabricated rows raises the detection rate to 81.4%. Although the fabricated series becomes more distinguishable from the genuine-data control distribution, the detection rate remains below the 95% threshold.

At 200 fabricated rows, corresponding to approximately 2.58% contamination, the detection rate rises to 97.0%. This exceeds the pre-defined 95% reliability criterion.

The smallest contamination reliably detected among the three levels tested is therefore **200 fabricated rows**, equivalent to approximately **2.58% of the contaminated sample**.

The experiment also shows that fabrication does not necessarily increase the chi-square statistic. Because the clean value-traded series already exhibits a substantial departure from Benford's Law, clustered fabricated observations can move the chi-square statistic in either direction. Detection is therefore based on whether the contaminated statistic falls outside the range normally generated by same-size additions of genuine observations, rather than simply whether the chi-square statistic increases.

# 5. Part C — Mini-index & the Point-in-Time Trap

This section constructs two three-stock indices using the cleaned AAPL, JPM, and XOM data over the full sample window.

The first index is initialized using market-capitalisation weights. The second allocates one-third of the initial portfolio value to each stock and then follows a buy-and-hold strategy without periodic rebalancing. Both indices are normalised to 100 on the first common trading day.

Adjusted closing prices are used for the performance paths so that stock splits and cash distributions do not create artificial return discontinuities. Initial market-capitalisation weights are estimated using historical shares outstanding and the corresponding split-adjusted closing prices.

## 5.1 Build the Two Indices (Requirement 1)

### Methodology

Initial market capitalisation is calculated as:

$$
\text{Market Cap}_{i,0}
=
\text{Price}_{i,0}
\times
\text{Shares Outstanding}_{i,0}.
$$

Historical shares outstanding are taken from the most recent observation available on or before the index start date.

An additional split-basis adjustment is required for AAPL because Yahoo Finance retrospectively adjusts historical `Close` prices for subsequent stock splits. As a result, the 2016 AAPL `Close` is expressed on the post-2020 4-for-1 split basis, while the historical shares-outstanding observation from 2016 remains on the contemporaneous pre-split basis.

Using these two quantities directly would therefore mix inconsistent units and understate the 2016 market capitalisation. To align the share-count basis with the retrospectively split-adjusted price basis, the 2016 AAPL shares outstanding are multiplied by the subsequent 4-for-1 split factor:

$$
\text{Split-basis adjusted shares}
=
\text{Historical shares}
\times
4.
$$

This adjustment does **not** mean that AAPL actually had 21.55 billion shares outstanding in 2016. It is only a basis-conversion step required because the historical price series has been retrospectively adjusted for the later stock split.

Equivalently, the same 2016 market capitalisation could be obtained by combining the original 2016 pre-split share count with the corresponding pre-split nominal share price.

No equivalent adjustment is required for JPM or XOM because no stock split occurs for these stocks within the sample window.

For the index paths, adjusted Close is used so that splits and cash
distributions are included. The resulting index levels are total-return-style
buy-and-hold portfolio series rather than daily-rebalanced official indices.


In [ ]:
# ------------------------------------------------------------
# Part C.1 — Initial Market-Capitalisation Weights
# ------------------------------------------------------------

INDEX_TICKERS = MAIN_TICKERS

# Use a robust project data path
DATA_DIR = (
    Path("../data")
    if Path.cwd().name == "code"
    else Path("data")
)

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 1. Build the common adjusted-price panel
# ------------------------------------------------------------

adj_close_panel = pd.concat(
    {
        ticker: cleaned_data[ticker]["Adj Close"]
        for ticker in INDEX_TICKERS
    },
    axis=1
)

adj_close_panel = (
    adj_close_panel
    .dropna()
    .sort_index()
)

first_date = adj_close_panel.index[0]
last_date = adj_close_panel.index[-1]

print(
    f"Index window: "
    f"{first_date.date()} to {last_date.date()}"
)

print(
    f"Common observations: "
    f"{len(adj_close_panel)}"
)


# ------------------------------------------------------------
# 2. Load cached historical shares outstanding
# ------------------------------------------------------------

def load_historical_shares(
    ticker,
    reference_date
):

    cache_path = (
        DATA_DIR
        / f"{ticker}_shares_outstanding.parquet"
    )

    # Submitted grading path: use the cached shares file only
    if cache_path.exists():

        shares_df = pd.read_parquet(
            cache_path
        )

        shares = shares_df[
            "Shares Outstanding"
        ]

    else:
        raise FileNotFoundError(
            f"Missing cached shares-outstanding file: {cache_path}. "
            "The submitted notebook requires the parquet cache and does not "
            "download data during grading."
        )

    shares.index = pd.to_datetime(
        shares.index
    )

    if shares.index.tz is not None:

        shares.index = (
            shares.index
            .tz_localize(None)
        )

    return shares.sort_index()


# ------------------------------------------------------------
# 3. Determine initial market capitalisation
# ------------------------------------------------------------

initial_market_cap_rows = []


for ticker in INDEX_TICKERS:

    df = cleaned_data[ticker].copy()

    df.index = pd.to_datetime(
        df.index
    )

    if df.index.tz is not None:

        df.index = (
            df.index
            .tz_localize(None)
        )


    # --------------------------------------------------------
    # Historical shares outstanding
    # --------------------------------------------------------

    shares_history = (
        load_historical_shares(
            ticker,
            first_date
        )
    )

    available_shares = (
        shares_history[
            shares_history.index
            <= first_date
        ]
    )

    if available_shares.empty:

        raise ValueError(
            f"No point-in-time shares observation "
            f"available on or before "
            f"{first_date.date()} for {ticker}."
        )

    shares_date = (
        available_shares.index[-1]
    )

    raw_shares = float(
        available_shares.iloc[-1]
    )


    # --------------------------------------------------------
    # Correct the historical share count for future splits
    #
    # Yahoo Close is retrospectively split-adjusted, while
    # historical shares outstanding are recorded on their
    # contemporaneous share-count basis.
    # --------------------------------------------------------

    split_events = df.loc[
        (
            df.index > first_date
        )
        &
        (
            df["Stock Splits"] > 0
        ),
        "Stock Splits"
    ]

    if len(split_events) > 0:

        future_split_factor = float(
            split_events.prod()
        )

    else:

        future_split_factor = 1.0


    split_adjusted_shares = (
        raw_shares
        * future_split_factor
    )


    # --------------------------------------------------------
    # Initial market cap
    # --------------------------------------------------------

    start_close = float(
        df.loc[
            first_date,
            "Close"
        ]
    )

    initial_market_cap = (
        start_close
        * split_adjusted_shares
    )


    initial_market_cap_rows.append({

        "Ticker":
            ticker,

        "Shares Observation Date":
            shares_date.date(),

        "Historical Shares":
            raw_shares,

        "Future Split Factor":
            future_split_factor,

        "Split-Basis Adjusted Shares":
            split_adjusted_shares,

        "Start Close":
            start_close,

        "Initial Market Cap":
            initial_market_cap
    })


initial_market_caps = pd.DataFrame(
    initial_market_cap_rows
).set_index(
    "Ticker"
)


# ------------------------------------------------------------
# 4. Calculate initial market-cap weights
# ------------------------------------------------------------

initial_market_caps[
    "Initial Weight"
] = (
    initial_market_caps[
        "Initial Market Cap"
    ]
    /
    initial_market_caps[
        "Initial Market Cap"
    ].sum()
)


display(
    initial_market_caps.style.format({

        "Historical Shares":
            "{:,.0f}",

        "Future Split Factor":
            "{:.2f}",

        "Split-Basis Adjusted Shares":
            "{:,.0f}",

        "Start Close":
            "${:,.2f}",

        "Initial Market Cap":
            "${:,.0f}",

        "Initial Weight":
            "{:.2%}"
    })
)

## 5.2 Plot, Report Weights, and Explain Drift (Requirement 2)

The market-cap-weighted series uses the initial split-basis market-capitalisation weights and adjusted-price relatives. It is therefore an initial-cap-weighted buy-and-hold portfolio. The equal-weighted series invests one-third in each stock initially and also does not rebalance. Adjusted Close is used for both paths, so the plotted levels are total-return-style series that include distributions.

The plot and the start-versus-end weight table below show how the fixed-share market-cap portfolio becomes increasingly concentrated in the strongest-performing constituent.


In [ ]:
# ------------------------------------------------------------
# Part C.2 — Build the Two Mini-Indices
# ------------------------------------------------------------

# ------------------------------------------------------------
# 1. Price relatives
# ------------------------------------------------------------

price_relatives = (
    adj_close_panel
    /
    adj_close_panel.iloc[0]
)


# ------------------------------------------------------------
# 2. Market-cap-weighted index
# ------------------------------------------------------------

initial_cap_weights = (
    initial_market_caps[
        "Initial Weight"
    ]
    .reindex(INDEX_TICKERS)
)


cap_position_values = (
    price_relatives
    .mul(
        initial_cap_weights,
        axis=1
    )
)


cap_weighted_index = (
    100
    * cap_position_values.sum(
        axis=1
    )
)


# Dynamic constituent weights
cap_weights_over_time = (
    cap_position_values
    .div(
        cap_position_values.sum(
            axis=1
        ),
        axis=0
    )
)


# ------------------------------------------------------------
# 3. Equal-initial-weight buy-and-hold index
# ------------------------------------------------------------

equal_initial_weights = pd.Series(
    1 / len(INDEX_TICKERS),
    index=INDEX_TICKERS
)


equal_position_values = (
    price_relatives
    .mul(
        equal_initial_weights,
        axis=1
    )
)


equal_buy_hold_index = (
    100
    * equal_position_values.sum(
        axis=1
    )
)


# ------------------------------------------------------------
# 4. Combine index levels
# ------------------------------------------------------------

mini_indices = pd.DataFrame({

    "Market-Cap Weighted":
        cap_weighted_index,

    "Equal Initial Weight Buy-and-Hold":
        equal_buy_hold_index
})


# Confirm both start at 100
print(
    "Starting index levels:"
)

display(
    mini_indices.head(1)
)


# ------------------------------------------------------------
# 5. Plot the two indices
# ------------------------------------------------------------

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    mini_indices.index,
    mini_indices[
        "Market-Cap Weighted"
    ],
    label="Market-Cap Weighted"
)

plt.plot(
    mini_indices.index,
    mini_indices[
        "Equal Initial Weight Buy-and-Hold"
    ],
    label="Equal Initial Weight Buy-and-Hold"
)

plt.axhline(
    100,
    linestyle="--",
    alpha=0.5
)

plt.title(
    "Three-Stock Mini-Indices"
)

plt.xlabel(
    "Date"
)

plt.ylabel(
    "Index Level"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 6. Start vs end market-cap weights
# ------------------------------------------------------------

start_weights = (
    cap_weights_over_time.iloc[0]
)

end_weights = (
    cap_weights_over_time.iloc[-1]
)


weight_drift = pd.DataFrame({

    "Start Weight":
        start_weights,

    "End Weight":
        end_weights
})


weight_drift[
    "Weight Drift (pp)"
] = (
    (
        weight_drift[
            "End Weight"
        ]
        -
        weight_drift[
            "Start Weight"
        ]
    )
    * 100
)


display(
    weight_drift.style.format({

        "Start Weight":
            "{:.2%}",

        "End Weight":
            "{:.2%}",

        "Weight Drift (pp)":
            "{:+.2f}"
    })
)


# ------------------------------------------------------------
# 7. Final index levels
# ------------------------------------------------------------

index_summary = pd.DataFrame({

    "Start Level":
        mini_indices.iloc[0],

    "End Level":
        mini_indices.iloc[-1],

    "Total Growth":
        (
            mini_indices.iloc[-1]
            /
            mini_indices.iloc[0]
            - 1
        )
})


display(
    index_summary.style.format({

        "Start Level":
            "{:.2f}",

        "End Level":
            "{:.2f}",

        "Total Growth":
            "{:.2%}"
    })
)

### Result

At the start of the index window, AAPL has the largest estimated market capitalisation and therefore receives the largest initial weight of 48.82%. XOM and JPM receive initial weights of 30.57% and 20.61%, respectively. The three weights sum to 100%.

The market-cap-weighted index ends at 881.13, above the equal-initial buy-and-hold index at 770.26. This difference is driven by the strong relative performance of AAPL: its portfolio weight rises from 48.82% to 73.81%. XOM falls from 30.57% to 10.13%, while JPM declines more moderately from 20.61% to 16.06%. The result illustrates weight drift: a fixed-share cap-weighted portfolio becomes increasingly concentrated in the strongest-performing constituent.

## 5.3 Point-in-Time Question (Requirement 3)

### (a) Why survivorship bias inflates the result

If a strategy is backtested over the past ten years using today's index constituents, the result is affected by survivorship bias. Today's constituents are companies that survived, remained large, or performed well enough to stay in the index. Delisted, bankrupt, acquired, or demoted companies are omitted, so the backtest removes many poor outcomes and can make the strategy look artificially successful.

### (b) What point-in-time membership means

Point-in-time membership means using the constituent list that was actually known and effective on each historical date, including additions and deletions with their effective dates. A point-in-time universe keeps companies that later disappeared in the historical sample and prevents today's information from leaking into the past.

### (c) One additional point-in-time trap

One additional trap is publication lag or restated fundamentals. For example, using a company's final restated Q4 earnings value on 31 December in a backtest leaks information that was only published weeks later and could create a falsely profitable signal.
